# Bank Distress Early-Warning Model — Feature Engineering
**MSDS 696 Data Science Practicum II · Oussama Ennaciri**

Picks up from `cleaning.ipynb` and adds the one thing the panel is missing: **history**.

Every row today is a single quarter — a still frame. But the EDA finding is that banks
decline over *two years* before they cross into distress, and a still frame cannot show a
decline by construction. Petropoulos et al. (2020) feed two years of lagged observations
per feature for exactly this reason; Cole & White (2012) and Correia/Luck/Verner (2024)
both single out asset growth.

This notebook adds, for each core measure, **how much it changed over the past 4 and 8
quarters** — and nothing else. No values are altered, no rows dropped.

Output: `panel_trend.parquet`, which `modeling.ipynb` reads.

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

PROCESSED = Path("..") / "data" / "processed"

panel = pd.read_parquet(PROCESSED / "panel_clean.parquet")
print(f"panel_clean: {panel.shape[0]:,} rows x {panel.shape[1]} cols")

## 1 · Which measures get a history

Not all 54 features — only the ones the literature treats as vitals, grouped by CAMELS
category. Adding a trend to every column would double the table for no gain: a bank's
state code does not have a trend, and the macro columns are identical for every bank in a
quarter, so their "change" would be one number repeated 19,000 times.

Two kinds of change, because the columns mean different things:

| Kind | Columns | Computed as | Reads as |
|---|---|---|---|
| **Ratios** (already %) | capital, asset quality, earnings, funding | today − a year ago | "fell 2.3 points" |
| **Dollar levels** | `ASSET`, `DEP` | today ÷ a year ago − 1 | "grew 14%" |

Subtracting dollar amounts would just measure bank size, which is why those two get a
growth rate instead.

In [ ]:
# Ratios -> change in percentage points.
RATIO_TREND = [
    # capital
    "RBCRWAJ", "RBC1AAJ", "EQV",
    # asset quality
    "NPERFV", "NCLNLSR", "ORER", "NTLNLSR", "LNATRESR",
    # earnings
    "ROA", "ROE", "NIMY", "EEFFR",
    # funding / liquidity  (the SVB blind spot)
    "DEPUNA", "BROR", "LNLSDEPR", "CHBALR",
]

# Dollar levels -> growth rate.
GROWTH_TREND = ["ASSET", "DEP"]

# 4 quarters = 1 year, 8 quarters = the two-year decline the EDA found.
HORIZONS = [4, 8]

print(f"{len(RATIO_TREND)} ratios + {len(GROWTH_TREND)} levels, at {HORIZONS} quarters back")
print(f"-> {(len(RATIO_TREND) + len(GROWTH_TREND)) * len(HORIZONS)} new columns")

## 2 · Look up the past by date, not by row position

The obvious way to get "a year ago" is `shift(4)` — go back four rows within each bank.
That is wrong whenever a bank has a missing quarter: the shift silently returns a filing
from five or six quarters back and labels it as one year. Only 17 rows in this panel sit
after such a gap, but a wrong lag is invisible once it is in the table, so it is worth
closing off entirely.

Instead, each row gets a **quarter number** (`year × 4 + quarter`), and the past is fetched
by joining the table to itself on `(bank, quarter number − 4)`. If the filing does not
exist, the result is blank — which is honest — rather than the wrong quarter.

**On leakage:** every lookup goes *backward*. Nothing here reads a quarter later than the
row it is attached to, so these columns are safe under the guardrail in `modeling.ipynb`.

In [ ]:
panel["_qi"] = panel["REPDTE"].dt.year * 4 + panel["REPDTE"].dt.quarter

# Confirm the join key is unique before relying on it.
assert not panel.duplicated(["CERT", "_qi"]).any(), "a bank has two filings in one quarter"

base = panel[["CERT", "_qi"] + RATIO_TREND + GROWTH_TREND]

for h in HORIZONS:
    # Shift the *past* forward by h quarters so it lines up with the row it belongs to.
    past = base.copy()
    past["_qi"] = past["_qi"] + h
    past = past.rename(columns={c: f"{c}__prev" for c in RATIO_TREND + GROWTH_TREND})

    panel = panel.merge(past, on=["CERT", "_qi"], how="left")

    for c in RATIO_TREND:
        panel[f"{c}_chg{h}q"] = panel[c] - panel[f"{c}__prev"]

    for c in GROWTH_TREND:
        # Guard against a zero denominator producing an infinite growth rate.
        prior = panel[f"{c}__prev"].replace(0, np.nan)
        panel[f"{c}_grow{h}q"] = panel[c] / prior - 1

    panel = panel.drop(columns=[f"{c}__prev" for c in RATIO_TREND + GROWTH_TREND])

panel = panel.drop(columns=["_qi"])

TREND_COLS = [c for c in panel.columns if "_chg" in c or "_grow" in c]
print(f"added {len(TREND_COLS)} trend columns -> {panel.shape[1]} total")

## 3 · How complete are they

A trend column is blank whenever the bank has no filing that far back — a young bank, or
one that joined the panel recently. That is a real gap, not an error, and it should stay
blank rather than be filled with a zero, which would read as "no change" and quietly claim
the bank was stable when nothing is known about it.

The 8-quarter columns are emptier than the 4-quarter ones for the same reason, and the
funding columns inherit the era gaps already documented in `cleaning.ipynb`
(`DEPUNA` starts ~1993).

In [ ]:
gaps = (panel[TREND_COLS].isna().mean() * 100).sort_values(ascending=False)

print(f"missing %:  min {gaps.min():.1f}   median {gaps.median():.1f}   max {gaps.max():.1f}\n")
print("emptiest five:")
print(gaps.head().round(1).to_string())
print("\nfullest five:")
print(gaps.tail().round(1).to_string())

## 4 · Do the trends actually carry signal

Before trusting these columns, check each one alone: how well does it separate "becomes
undercapitalized within a year" from "stays fine"? Scored by AUC, where 0.5 is a coin flip.

**Computed on the training years only (1990–2015).** Ranking features on the full panel
would let the test period influence which features get kept — the feature-selection
leakage noted at the end of `modeling.ipynb`.

In [ ]:
train_rows = panel[(panel["onset_4q"].notna()) & (panel["REPDTE"] <= "2015-12-31")]


def solo_auc(col: str) -> tuple[float, int]:
    """AUC of one column on its own. Direction-free: a feature that predicts perfectly
    backwards is just as informative as one that predicts perfectly forwards."""
    s = train_rows[[col, "onset_4q"]].dropna()
    if s["onset_4q"].nunique() < 2:
        return float("nan"), len(s)
    a = roc_auc_score(s["onset_4q"], s[col])
    return max(a, 1 - a), len(s)


scores = pd.DataFrame(
    [(c, *solo_auc(c)) for c in TREND_COLS], columns=["feature", "auc", "n"]
).sort_values("auc", ascending=False)

# The capital ratio level, for reference — the bar every trend feature is measured against.
level_auc, _ = solo_auc("RBCRWAJ")

print(f"capital ratio LEVEL (reference): {level_auc:.3f}\n")
print(scores.head(12).to_string(index=False, float_format=lambda v: f"{v:.3f}"))

### What the scores say

The trends carry real signal on their own — change in bad loans and change in capital both
land well above a coin flip — but **none of them beats the capital ratio's level**.

That is the expected result, and it is worth stating plainly rather than glossing: the
target is defined by capital crossing a threshold, so the current level sits mechanically
close to the answer. A bank at 9% is one bad quarter from 8%.

The trends earn their place only if they add something the level does not already contain —
which is a question about the *combination*, not about any single column, and so cannot be
answered here. It gets answered in `modeling.ipynb`, by whether the model beats the
capital-ratio benchmark.

## 5 · Save

No rows dropped, no existing value altered — 36 columns added.
`modeling.ipynb` reads this file.

In [ ]:
out = PROCESSED / "panel_trend.parquet"
panel.to_parquet(out, index=False)

print(f"saved {out.name}: {panel.shape[0]:,} rows x {panel.shape[1]} cols")
print(f"  {len(TREND_COLS)} trend columns added")